In [1]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import sentencepiece as spm

In [2]:
spp = spm.SentencePieceProcessor(model_file='../med_fine_sp.model')
vocab_size = spp.get_piece_size()
print(vocab_size)

50257


In [3]:
@dataclass
class config:
    block_size:int=1024
    vocab_size:int=spp.get_piece_size()
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    n_dim: int = 768  # for layernorm
    dropout: float = 0.3
    bias: bool = True

In [4]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.lay_1=nn.Linear(config.n_embd,4*config.n_embd)
        self.gelu=nn.GELU(approximate='tanh')
        self.out=nn.Linear(4*config.n_embd,config.n_embd)
    
    def forward(self,x):
        x=self.lay_1(x)
        x=self.gelu(x)
        x=self.out(x)
        return x


In [5]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_drop = nn.Dropout(config.dropout)
        self.proj_drop = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x).reshape(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Use scaled_dot_product_attention with causal mask
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.attn_drop.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().reshape(B, T, C)
        y = self.proj(y)
        y = self.proj_drop(y)
        return y


In [6]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # ⭐ FLASH ATTENTION HERE (PyTorch >= 2.0 support check)
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))
            
    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # ⭐ FLASH ATTENTION HERE (efficient attention using Flash Attention CUDA kernels)
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


In [7]:
# Using built-in nn.LayerNorm for stability and efficiency
# Remove custom LayerNorm - not needed

In [8]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = Attention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln_1(x)))
        x = x + self.dropout(self.mlp(self.ln_2(x)))
        return x

In [ ]:
class GPT(nn.Module):


    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss
    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]
    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [10]:
import torch; print(f'CUDA available: {torch.cuda.is_available()}'); print(f'Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None' }')

CUDA available: True
Device name: NVIDIA GeForce RTX 4050 Laptop GPU


In [11]:
# Training Loop with Tokenization
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Initialize model
cfg = config()
model = GPT(cfg)
model = model.to(device)

# Create optimizer
optimizer = model.configure_optimizers(
    weight_decay=0.01,
    learning_rate=1e-4,
    betas=(0.9, 0.95),
    device_type=device
)
print("✓ Optimizer configured")

# Load and tokenize data
print("\nLoading and tokenizing data...")
try:
    all_tokens = []
    num_records = 0
    max_records = 100  # Limit for testing
    
    # Read JSONL file
    with open('../pmc_articles.jsonl', 'r') as f:
        for line in f:
            if num_records >= max_records:
                break
            try:
                record = json.loads(line)
                # Extract text - adjust key based on your JSON structure
                text = record.get('text', '') or record.get('content', '') or record.get('abstract', '')
                
                if text:
                    # Tokenize with SentencePiece
                    tokens = spp.encode(text)
                    all_tokens.extend(tokens)
                    num_records += 1
                    
                    if num_records % 20 == 0:
                        print(f"  Processed {num_records} records, {len(all_tokens)} tokens so far")
            except Exception as e:
                continue
    
    print(f"✓ Loaded {num_records} records, {len(all_tokens)} total tokens")
    
    # Convert to tensor
    all_tokens = torch.tensor(all_tokens, dtype=torch.long, device=device)
    print(f"✓ Tokens tensor shape: {all_tokens.shape}")
    
except Exception as e:
    print(f"✗ Error loading data: {e}")
    raise

# Create batches from token sequence
print("\nCreating batches...")
batch_size = 4
seq_length = 128

train_data = []
for i in range(0, len(all_tokens) - seq_length, batch_size * seq_length):
    batch_tokens = []
    for j in range(batch_size):
        start = i + j * seq_length
        end = start + seq_length + 1
        
        if end <= len(all_tokens):
            seq = all_tokens[start:end]
            batch_tokens.append(seq)
    
    if len(batch_tokens) == batch_size:
        batch = torch.stack(batch_tokens)
        train_data.append(batch)

print(f"✓ Created {len(train_data)} batches of shape (batch_size=4, seq_length=129)")

# Training loop
print("\n" + "="*50)
print("Starting Training with Real Tokenized Data")
print("="*50)

model.train()
num_epochs = 1

try:
    for epoch in range(num_epochs):
        total_loss = 0
        for batch_idx, batch_tokens in enumerate(train_data):
            input_ids = batch_tokens[:, :-1]
            targets = batch_tokens[:, 1:]
            
            # Forward pass
            logits, loss = model(input_ids, targets)
            
            # Backward pass
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            loss_val = loss.item()
            total_loss += loss_val
            
            if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == len(train_data):
                print(f"Batch {batch_idx+1}/{len(train_data)} | Loss: {loss_val:.4f}")
        
        avg_loss = total_loss / len(train_data)
        print(f"\nEpoch {epoch+1} | Avg Loss: {avg_loss:.4f}")
    
    print("\n✓ Training completed successfully with real tokenized data!")

except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

Using device: cuda
number of parameters: 123.65M
num decayed parameter tensors: 50, with 124,318,464 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: True
✓ Optimizer configured

Loading and tokenizing data...
  Processed 20 records, 255470 tokens so far
  Processed 40 records, 507308 tokens so far
  Processed 60 records, 729474 tokens so far
  Processed 80 records, 960001 tokens so far
  Processed 100 records, 1236101 tokens so far
✓ Loaded 100 records, 1236101 total tokens
✓ Tokens tensor shape: torch.Size([1236101])

Creating batches...
✓ Created 2414 batches of shape (batch_size=4, seq_length=129)

Starting Training with Real Tokenized Data
Batch 5/2414 | Loss: 10.0149
Batch 10/2414 | Loss: 9.7485
Batch 15/2414 | Loss: 9.8956
Batch 20/2414 | Loss: 9.2971
Batch 25/2414 | Loss: 9.1395
Batch 30/2414 | Loss: 9.2803
Batch 35/2414 | Loss: 8.7771
Batch 40/2414 | Loss: 8.4888
Batch 45/2414 | Loss: 7.9370
Batch 50/2414 | Loss: 8.0399
Batch 55/2414

In [13]:
# Generate Samples from Trained Model with Custom Prompts
print("\n" + "="*50)
print("Text Generation with Custom Prompts")
print("="*50)

model.eval()

# Generation parameters
max_new_tokens = 100
temperature = 0.7
top_k = 40

# Define custom prompts
prompts = [
    "The study shows that",
    "In medical research,",
    "The results indicate that"
]

try:
    with torch.no_grad():
        for prompt_id, prompt_text in enumerate(prompts):
            print(f"\n--- Prompt {prompt_id + 1}: \"{prompt_text}\" ---")
            
            # Tokenize the prompt
            seed_tokens_list = spp.encode(prompt_text)
            seed_tokens = torch.tensor(seed_tokens_list, dtype=torch.long, device=device).unsqueeze(0)
            
            print(f"Seed tokens ({len(seed_tokens_list)}): {seed_tokens_list}")
            
            # Generate new tokens
            generated = model.generate(
                seed_tokens,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k
            )
            
            # Decode tokens to text
            token_list = generated[0].cpu().numpy().tolist()
            decoded_text = spp.decode(token_list)
            
            print(f"\nGenerated text:\n{decoded_text}\n")
    
    print("="*50)
    print("✓ Text generation completed!")
    print("="*50)

except Exception as e:
    print(f"✗ Error generating text: {e}")
    import traceback
    traceback.print_exc()

# INTERACTIVE MODE - Uncomment to use custom input
"""
print("\n" + "="*50)
print("Interactive Mode - Enter your own prompt")
print("="*50)

user_prompt = input("\nEnter a prompt: ").strip()

if user_prompt:
    with torch.no_grad():
        seed_tokens_list = spp.encode(user_prompt)
        seed_tokens = torch.tensor(seed_tokens_list, dtype=torch.long, device=device).unsqueeze(0)
        
        generated = model.generate(
            seed_tokens,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k
        )
        
        token_list = generated[0].cpu().numpy().tolist()
        decoded_text = spp.decode(token_list)
        
        print(f"\nYour prompt: \"{user_prompt}\"")
        print(f"\nGenerated continuation:\n{decoded_text}")
"""


Text Generation with Custom Prompts

--- Prompt 1: "The study shows that" ---
Seed tokens (4): [339, 918, 2247, 338]

Generated text:
The study shows that the same model in the this time the blood and an A and the number to the increase in the the blood the same it in the number. In the results of the a most the patients in the present in the blood the high in the number in the brain in the data is the samples in the control to the first on the three the high in the data in the be# be the data in the ability to the number, well to a effect in the this the three the study in the used


--- Prompt 2: "In medical research," ---
Seed tokens (4): [455, 1933, 1015, 49302]

Generated text:
In medical research, the EC, the A., Liom-R. *N-S., K.R., Zhang J. J., Z.. **R., S.. **E.. **j.A.H.,sed of the study.B., B., S. J.. **T., Wang**. *20**. Med. *J. *F., K. *N., Zh.* (20.L., Al.* (2019. DOI: 10.1016/j.S., K.A


--- Prompt 3: "The results indicate that" ---
Seed tokens (4): [339, 1706, 4863, 3

'\nprint("\n" + "="*50)\nprint("Interactive Mode - Enter your own prompt")\nprint("="*50)\n\nuser_prompt = input("\nEnter a prompt: ").strip()\n\nif user_prompt:\n    with torch.no_grad():\n        seed_tokens_list = spp.encode(user_prompt)\n        seed_tokens = torch.tensor(seed_tokens_list, dtype=torch.long, device=device).unsqueeze(0)\n\n        generated = model.generate(\n            seed_tokens,\n            max_new_tokens=max_new_tokens,\n            temperature=temperature,\n            top_k=top_k\n        )\n\n        token_list = generated[0].cpu().numpy().tolist()\n        decoded_text = spp.decode(token_list)\n\n        print(f"\nYour prompt: "{user_prompt}"")\n        print(f"\nGenerated continuation:\n{decoded_text}")\n'